## Pydantic을 이용해 Tool의 입력값 정의

In [3]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True,
)

## Pydantic으로 정의한 입력값을 Tool에 사용 / Tool을 LLM에 등록

In [4]:
import sys
sys.path.append("..")

from langchain_tool_functions import tools, tool_dict

llm_with_tools = llm.bind_tools(tools)

## 사용자의 질문을 LLM에 전달

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.")
]

messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)


print("=== 응답 ===")
print(response.content)

print("=== 툴 콜 ===")
print(response.tool_calls)

content=[{'id': 'rs_0c1460543379df42006aa2eb4acfa487d095429952c9181a06', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqoutLFNLdaSEmDtx0OeeOd8WlZJhYVJ7zK0gz-YnVztDaXaXP20omKSIKAIfh8-g-F2XTb3oTULd-v770RDF0nnU3UPpX_CQO-9dIRyE7BG3m_r1lKibRPNXc3PEn1l-vHGA-jU6e5grIZTiaoMYH_4WOxL7BcboNwRHQWVYFkeD8q3CuOHxeUMQKNEFUBEMlIqYL88hmxIlV71XrFY_shQT3QpgovYYr5AtU1AEzb9kc0ACYx42btGHqf70PwwXxSDJNoMhQWLF1OUe5QJpbgALeA6XKkhU0p29p7u8LMEWC2N6ZOZd7jNDW9UU6SPIb4gk18QCt44gYvS-d1Zs5tbDgQ-wwR9LY1SLqe9nNP6dfmBGX22B3jOylq-0PFYsVawoeKsJ8zXAknUft-wTwK3XS0pXKvsCG3aJHF3qhW6Bm-X7QzTmIwPyNca3MrZcpx84xSLg3-H3vv-1RDtNLf64Tvg0Y2tqYkxDzSfj8TiTUWfIVDQ_vSb9HRCHCugNrgMHyXLYvpUNXG0W5fNguYVGotfIdCBYzXxFWXXHXsdRNcnvCxayB-Pbpgi0a8jc3LcbYTJI2szLj_sjSrlg0LdLnZDzJKprwaS9q-LT3zRYvR_uBKDz2gy8m8SmoPdnW0ZyYRjKAyQ-lHDt3QiRebXhCS1kX2euCooOWbbtb6dbrw7yDAZ_qopRDM0nyrsyN3YJPom9PdAvqGGWMbuivTFFervnpOMO-l37StyJkpzXSM23YxDecZApIx3YRYQMJ7kt1ocuzd0QJlg5nO7VMOSYSE_KtD7jJwaCe9_mTcVNE3WfLOouqJIFjlkthmSilC7hQDnI6kuLKgU

- tool_calls가 get_yf_stock_history로 나온 것 = 툴 선택 성공

## LLM이 선택한 Tool 실행

In [6]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

print("=== 툴 결과 ===")
print(tool_msg.content)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-08-10 00:00:00-04:00 | 326.6  | 332.05 | 326.15 |  330.88 | 2.50038e+07 |           0 |              0 |\n| 2026-08-11 00:00:00-04:00 | 332.8  | 336.2  | 329.53 |  332.81 | 2.33452e+07 |           0 |              0 |\n| 2026-08-12 00:00:00-04:00 | 335    | 335.5  | 323.64 |  327.51 | 2.86989e+07 |           0 |              0 |\n| 2026-08-13 00:00:00-04:00 | 327.2  | 341.64 | 325.24 |  339.96 | 3.47083e+07 |           0 |              0 |\n| 2026-08-14 00:00:00-04:00 | 342.33 | 351.26 | 335.33 |  342.27 | 4.54371e+07 |           0 |              0 |\n| 2026-08-17 00:00:00-04:00 | 340.69 | 345.45 | 337.48 |  339.3  | 2.60371e+07 |           0 |              0 |\n| 2026-08-18 00:00:00-04:0

- selected_tool = ... → GPT가 선택한 함수(get_yf_stock_history)를 찾음
- selected_tool.invoke(tool_call) → 그 함수에 GPT가 만든 인자를 넣어 실제 실행함
- messages.append(tool_msg) → 실행 결과를 대화 기록에 넣어서, 다음 LLM 호출이 결과를 보고 최종 답변하도록 준비함

## Tool의 결과를 LLM에게 전달 / 최종 답변 생성

In [7]:
response = llm_with_tools.invoke(messages)

final_answer = next(
    item["text"]
    for item in response.content
    if item.get("type") == "text"
)

print(final_answer)

테슬라(TSLA)는 **한 달 전보다 올랐습니다.**

- 한 달 전 종가: **330.88달러**
- 최근 종가: **366.81달러**
- 변동: **+35.93달러, 약 +10.9%**

※ 조회된 거래일 종가 기준입니다.


```txt
👤 사용자
   │
   │ "테슬라 한 달 전보다 올랐나?"
   ▼
🤖 LLM
   │
   │ tool_call 생성
   │ get_yf_stock_history
   │ ticker=TSLA, period=1mo
   ▼
🐍 Python
   │
   │ selected_tool.invoke(tool_call)
   ▼
📈 yfinance
   │
   │ 주가 데이터 조회
   ▼
📦 ToolMessage
   │
   │ messages.append(tool_msg)
   ▼
🤖 LLM
   │
   │ llm_with_tools.invoke(messages)
   ▼
💬 최종 답변
   "테슬라는 한 달 전보다 약 10.1% 올랐습니다."
```